In [1]:
from config import *
from pdf_loader import read_pdf_text
from chunking import split_sections,chunk_text
from embeddings import load_embedder, embed_chunks
from db import init_db, upsert_chunks
from retrieval import retrieve_topk
from rerank import rerank_results
from chunking import build_chunks

import numpy as np

In [2]:
from db import delete_all_chunks,count_chunks,preview_chunks
count_chunks(PG_CONN_STR)
preview_chunks(PG_CONN_STR)

📦 Total stored chunks: 3480

🔎 Stored Chunks Preview

DOC: BaraaCV
SECTION: contents
. 9 leaky units and other strategies for multiple time scales.... 406 10. 10 the long short - term memory and other gated rnns...... 408 10. 11 optimization for long - term dependencies............. 4
----------------------------------------
DOC: BaraaCV
SECTION: contents
functions f f : a b → the function with domain and range a b f g f g [UNK] composition of the functions and f ( ; ) x θ a function of x parametrized by θ. ( sometimes we write f ( x ) and omit the arg
----------------------------------------
DOC: BaraaCV
SECTION: contents
.............. 425 11. 3 determining whether to gather more data............ 426 11. 4 selecting hyperparameters...................... 427 11. 5 debugging strategies......................... 436 11. 6
----------------------------------------
DOC: BaraaCV
SECTION: contents
............ 443 12. 2 computer vision........................... 452 12. 3 speech recognition..

In [3]:
print("🔎 Loading embedding model...")
embedder = load_embedder(EMBED_MODEL_NAME)

dim = embedder.get_sentence_embedding_dimension()

from db import init_db

# =========================================
# 2️ INIT DATABASE
# =========================================

print("🧱 Initializing database...")
init_db(PG_CONN_STR, dim)
doc_name = "BaraaCV"


🔎 Loading embedding model...


c:\Users\baraa\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


🧱 Initializing database...


In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=" ", # https://huggingface.co/settings/tokens
)
# ===============================
# Function: Generate Text
# =============================== 
# https://huggingface.co/Qwen/Qwen3-Coder-Next 

def generate_text(context, question,
                  model="Qwen/Qwen3-Coder-Next:novita",
                  temperature=0.2,
                  max_tokens=1024):

    prompt = f"""
    You are an AI assistant that answers questions strictly using the provided CONTEXT.

    Your task is to answer the QUESTION using ONLY the information that appears in the CONTEXT.

    STRICT RULES:
    - Use ONLY the information inside the CONTEXT.
    - Do NOT use prior knowledge or external information.
    - Do NOT guess or infer information that is not explicitly stated.
    - If the answer is not clearly present in the CONTEXT, respond exactly with:
    "The answer is not found in the provided context."
    - Do NOT add explanations that are not supported by the CONTEXT.
    - Prefer quoting or closely paraphrasing the CONTEXT.

    Answer style:
    - Keep the answer clear and concise.
    - Use bullet points when listing multiple facts.

    Output format:

    Answer:
    <answer based strictly on the context>

    Source:
    <section name if available>

    --------------------------------

    CONTEXT:
    {context}

    QUESTION:
    {question}
    """

    try:

        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )

        return completion.choices[0].message.content.strip()

    except Exception as e:

        return f"Error generating response: {str(e)}"

In [ ]:
from IPython.display import display, HTML
import numpy as np


def ask_rag_question(
    question,
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text,
    TOP_K=5
):

    # =========================================
    # 1️⃣ EMBEDDING
    # =========================================
    qvec = embedder.encode(
        [question],
        normalize_embeddings=True
    )[0].astype(np.float32)


    # =========================================
    # 2️⃣ VECTOR SEARCH
    # =========================================
    results = retrieve_topk(
        PG_CONN_STR,
        doc_name,
        qvec,
        TOP_K
    )
    # =========================================
    # 3️⃣ RE-RANK (مستقبلا)
    # =========================================
    reranked = results

    # =========================================
    # 4️⃣ BUILD CONTEXT
    # =========================================
    context = "\n\n".join([
        f"SECTION: {r[0]}\n{r[1]}"
        for r in reranked[:3]
    ])


    # =========================================
    # 5️⃣ GENERATE ANSWER
    # =========================================
    answer = generate_text(context, question)


    # =========================================
    # 6️⃣ FORMAT ANSWER
    # =========================================
    formatted_answer = answer.replace("- ", "• ").replace("\n", "<br>")


    # =========================================
    # 7️⃣ BUILD HTML
    # =========================================
    html = f"""
    <div style="border:2px solid #1976D2; padding:20px; border-radius:10px; font-family:Arial;">

    <h2>🔎 RAG Query Result</h2>

    <p><b>Question:</b> {question}</p>

    <hr>

    <h3>🤖 Generated Answer</h3>

    <div style="
        background:#e3f2fd;
        padding:16px;
        border-radius:10px;
        margin-bottom:20px;
        border-left:6px solid #1976D2;
        font-size:15px;
        line-height:1.6;
        box-shadow:0 2px 6px rgba(0,0,0,0.08);
    ">

    <div style="font-weight:600; margin-bottom:8px;">
    AI Response
    </div>

    <div>
    {formatted_answer}
    </div>

    </div>

    <hr>

    <h3>🏆 Top Retrieved Chunks</h3>
    """


    for r in reranked[:3]:

        section, content, sim, score = r

        html += f"""
        <div style="border:1px solid #ddd; padding:12px; margin-bottom:12px; border-radius:6px;">

        <p><b>Section:</b> {section}</p>
        <p><b>Vector Similarity:</b> {round(sim,3)}</p>
        <p><b>Final Score:</b> {round(score,3)}</p>

        <div style="background:#f5f5f5; padding:10px; border-radius:5px; font-size:14px;">
        {content[:400]}...
        </div>

        </div>
        """

    html += "</div>"

    display(HTML(html))

    return reranked, answer

In [6]:
reranked, answer = ask_rag_question(
    "What are the main parts of the Deep Learning book?",
    embedder,
    PG_CONN_STR,
    doc_name,
    retrieve_topk,
    rerank_results,
    generate_text
)